# Late Interaction using ColBert
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring serverside.<br>
However, turbopuffer doesn't support multi-vector documents today.

Since turbopuffer does not natively support multi-vector documents, a practical ColBERT implementation today is a two-stage approximation:
* dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation desciions:
* At index time, ColBERT is run on Every document and token vectors vectors are stored as JSON attribute on each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer trturns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.
Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors trabel with the document, no lookups or joins.
* Storing JSOB blobs is cheaper in turbopuffer's storage model.


### Step 1: Setup
Imports, load .env, initialize turbopuffer client

In [39]:
import turbopuffer
from dotenv import load_dotenv
import os

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-central1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-late-interaction')

### Step 2: Dataset
Load [Quora Question Paris](https://huggingface.co/datasets/sentence-transformers/quora-duplicates) from HuggingFace

In [12]:
import pandas as pd

df_qpair = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair/train-00000-of-00001.parquet")
df_qpair.count()

anchor      149263
positive    149263
dtype: int64

In [13]:
df_qpair.head(100)

,anchor,positive
0,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan..."
1,How can I be a good geologist?,What should I do to be a great geologist?
2,How do I read and find my YouTube comments?,How can I see all my Youtube comments?
3,What can make Physics easy to learn?,How can you make physics easy to learn?
4,What was your first sexual experience like?,What was your first sexual experience?
...,...,...
95,Will Modi win in 2019?,Can Narendra Modi become Prime Minister of Ind...
96,"What exactly is the ""Common Core Initiative/St...",What are the pros and cons of the Common Core ...
97,How do I choose a journal to publish my paper?,Where do I publish my paper?
98,What are your New Year's resolutions for 2017?,What is your creative New Year's resolution fo...


The dataset has ~150K questions pairs - `anchor` and `positive`. <br>We will:
* take a subset of 1000 questions pairs
* use `positive` to build corpus, create dense embeddings + `token_vector` attribute in turbopuffer
* use a few `anchor` questions to compare the search performance of **dense retreival** vs **late interaction**

Succes metric would be "did the matching positive for given anchor rank #1".

In [38]:
df_corpus = df_qpair['positive'][:1000]
print(f"Dataframe shape: {df_corpus.shape}")
print(f"Corpus length is {df_corpus.count()} \nCorpus Sample:")
df_corpus.head()

Dataframe shape: (1000,)
Corpus length is 1000 
Corpus Sample:


0    I'm a triple Capricorn (Sun, Moon and ascendan...
1            What should I do to be a great geologist?
2               How can I see all my Youtube comments?
3              How can you make physics easy to learn?
4               What was your first sexual experience?
Name: positive, dtype: str

### Step 3: Embedding
Load ColBERT model via fastembed, generate dense vector and token vectors per document.<br>
> We choose [`fastembed`](https://huggingface.co/ferrisS/harrier-oss-v1-270m-fastembed) because it is a lightweight embedding library with native late interaction support, and runs on CPU.

In [30]:
from fastembed import LateInteractionTextEmbedding

model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

In [32]:
sample = df_corpus[0]
vector = list(model.embed([sample]))[0]

print(sample)
print(vector.shape)

I'm a triple Capricorn (Sun, Moon and ascendant in Capricorn) What does this say about me?
(26, 128)


In [34]:
# one single vector to represent the whole document
dense_vector = vector.mean(axis=0)

# matrix of 128-dimensional vector of all tokens in the document.
token_vectors = vector.tolist()

print(f"Dense vector shape: {dense_vector.shape}")
print(f"Token matrix: {len(token_vectors)} tokens x {len(token_vectors[0])} dims")

Dense vector shape: (128,)
Token matrix: 26 tokens x 128 dims


In [43]:
# Loop over all 1000 corpus questions, generate dense and token vectors.

records = []

for i in range(len(df_corpus)):
    text = df_corpus[i]
    embedding = list(model.embed([text]))[0]

    # one single vector to represent the whole document
    dense_vector = embedding.mean(axis=0).tolist()

    # matrix of 128-dimensional vector of all tokens in the document.
    token_vectors = embedding.tolist()

    record = {
        "id": i + 1,
        "vector": dense_vector,
        "token_vectors": token_vectors,
        "text": text
    }

    records.append(record)
print(f"Embedded {len(records)} documents")



Embedded 1000 documents
